
# Football Scouting Project – ETL Pipeline

This notebook shows the ETL process used for our football scouting dataset.
applied to a football scouting dataset.

The goal is to clean and prepare the data so it can be used for analysis., including:
- Extraction strategies
- Staging areas
- Data quality issues
- Basic and advanced transformations
- Loading strategies

Dataset context: Football player scouting and performance analytics.


In [ ]:

import pandas as pd
import numpy as np



## 1. Extract Phase

In this project, the data is extracted from an Excel file.:
- Data is extracted from an external Excel file
- Represents a periodic snapshot of scouting data

The extracted data is first loaded into a **staging area**.


In [ ]:

# Offline extraction from Excel source
raw_df = pd.read_excel("football_scouting_dataset_10000_players.xlsx")

# Inspect raw structure
raw_df.shape, raw_df.head()



### Extraction Quality Checks
At this stage we only *inspect* data quality issues without fixing them yet.


In [ ]:

# Check missing values
raw_df.isnull().sum().sort_values(ascending=False).head(10)


In [ ]:

# Check duplicate primary keys
raw_df.duplicated(subset=["player_id"]).sum()



## 2. Staging Area


In [ ]:

staging_df = raw_df.copy()
staging_df.shape



## 3. Data Transformation



### 3.1 Remove Duplicated Data


In [ ]:

staging_df = staging_df.drop_duplicates(subset=["player_id"])



### 3.2 Mapping Null Values

Business rules:
- Injury days → missing values mapped to 0
- Salary → missing values mapped to median salary


In [ ]:

staging_df["injury_days_per_season"] = staging_df["injury_days_per_season"].fillna(0)
staging_df["monthly_salary_k_eur"] = staging_df["monthly_salary_k_eur"].fillna(
    staging_df["monthly_salary_k_eur"].median()
)



### 3.3 Format Conversion

- Convert `player_id` to string


In [ ]:

staging_df["player_id"] = staging_df["player_id"].astype(str)



## 4. Data Transformation – Advanced Processing



### 4.1 Splitting Columns

Split `player_name` into first and last name for analytical flexibility.


In [ ]:

name_split = staging_df["player_name"].str.split(" ", n=1, expand=True)
staging_df["first_name"] = name_split[0]
staging_df["last_name"] = name_split[1]



### 4.2 Filtering Rows

Apply business constraints:
- Age between 16 and 40
- Players must have played minutes
- Market value must be positive


In [ ]:

staging_df = staging_df[
    (staging_df["age"].between(16, 40)) &
    (staging_df["minutes_played"] > 0) &
    (staging_df["market_value_million_eur"] > 0)
]



### 4.3 Derived Columns (KPI Engineering)


In [ ]:

staging_df["goals_per90"] = staging_df["goals"] / (staging_df["minutes_played"] / 90)
staging_df["assists_per90"] = staging_df["assists"] / (staging_df["minutes_played"] / 90)
staging_df["disciplinary_index"] = staging_df["yellow_cards"] + 3 * staging_df["red_cards"]
staging_df["value_efficiency"] = (
    staging_df["goals"] + staging_df["assists"]
) / staging_df["market_value_million_eur"]
staging_df["potential_upside"] = staging_df["potential"] - staging_df["overall_rating"]



### 4.4 Aggregation Example

Aggregate average market value and performance by position.
This simulates **fact aggregation** used in data warehouses.


In [ ]:

position_agg = staging_df.groupby("position").agg({
    "market_value_million_eur": "mean",
    "goals_per90": "mean",
    "assists_per90": "mean"
}).reset_index()

position_agg



### 4.5 Data Validation

Final validation before loading:
- No negative KPIs
- No missing primary keys


In [ ]:

assert staging_df["player_id"].isnull().sum() == 0
assert (staging_df["goals_per90"] >= 0).all()
assert (staging_df["assists_per90"] >= 0).all()



## 5. Load Phase

Load Step:
- **Initial full load**
- Output written as CSV

In [ ]:

staging_df.to_csv("football_scouting_clean_dataset.csv", index=False)



### Load Summary
- Data successfully loaded
- Dataset is analytics-ready
- ETL process completed
